In [1]:
# Cell 1 — Konfiguration
import json, time, os, calendar
from pathlib import Path
from datetime import datetime, date, timedelta, timezone
import requests
from requests.auth import HTTPBasicAuth

NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "script" else NOTEBOOK_DIR
RAW_DIR = PROJECT_ROOT / "raw" / "freshdesk"
RAW_DIR.mkdir(parents=True, exist_ok=True)

CREDENTIALS_PATH = PROJECT_ROOT / "credentials" / "Freshdesk_API-key.txt"
FRESHDESK_DOMAIN = os.environ.get("FRESHDESK_DOMAIN", "intersolia")
FRESHDESK_API_URL = f"https://{FRESHDESK_DOMAIN}.freshdesk.com/api/v2/tickets"

# Backfill-intervall — ändra dessa
BACKFILL_FROM = date(2025, 5, 1)
BACKFILL_TO   = datetime.now(timezone.utc).date()

PAGE_SIZE     = 100
MAX_PAGES     = 300   # Freshdesks hårda gräns
MAX_RETRIES   = 5
RETRY_BACKOFF = 2
USER_AGENT    = "puttaren-agent/1.0"

print(f"PROJECT_ROOT:   {PROJECT_ROOT}")
print(f"RAW_DIR:        {RAW_DIR}")
print(f"Backfill range: {BACKFILL_FROM} → {BACKFILL_TO}")

PROJECT_ROOT:   C:\Users\michael.brostrom\Documents\GitHub\statistik-freshdesk-linear
RAW_DIR:        C:\Users\michael.brostrom\Documents\GitHub\statistik-freshdesk-linear\raw\freshdesk
Backfill range: 2025-05-01 → 2026-06-04


In [2]:
# Cell 2 — API-nyckel
def read_api_key() -> str:
    key = os.environ.get("FRESHDESK_API_KEY", "").strip()
    if key:
        print("Använder FRESHDESK_API_KEY från miljövariabel.")
        return key
    print(f"Läser API-nyckel från: {CREDENTIALS_PATH}")
    if not CREDENTIALS_PATH.exists():
        raise FileNotFoundError(f"API-nyckel saknas: {CREDENTIALS_PATH}")
    key = CREDENTIALS_PATH.read_text(encoding="utf-8").strip()
    if not key:
        raise ValueError("API-nyckeln är tom.")
    return key

In [3]:
# Cell 3 — Fältextraktion, fetch och chunk-logik
def extract_fields(ticket: dict) -> dict:
    return {
        "id":         ticket.get("id"),
        "subject":    ticket.get("subject"),
        "status":     ticket.get("status"),
        "priority":   ticket.get("priority"),
        "created_at": ticket.get("created_at"),
        "updated_at": ticket.get("updated_at"),
        "due_by":     ticket.get("due_by"),
        "group_id":   ticket.get("group_id"),
        "product_id": ticket.get("product_id"),
    }


def month_windows(from_date: date, to_date: date) -> list[tuple[date, date]]:
    """Returns (window_start, window_end_exclusive) pairs, one per month."""
    windows = []
    cur = from_date.replace(day=1)
    while cur <= to_date:
        last_day = calendar.monthrange(cur.year, cur.month)[1]
        nxt = (cur.replace(day=last_day) + timedelta(days=1))
        windows.append((cur, min(nxt, to_date + timedelta(days=1))))
        cur = nxt
    return windows


def fetch_window(api_key: str, window_start: date, window_end: date) -> list:
    """
    Hämtar tickets med updated_at i [window_start, window_end).
    Frågar API:t med updated_since=window_start och filtrerar klientsidan.
    Kastar ValueError om 300-sidors gränsen nås (fönstret är för stort).
    """
    auth = HTTPBasicAuth(api_key, "X")
    headers = {"Content-Type": "application/json", "User-Agent": USER_AGENT}
    base_params = {
        "per_page":      PAGE_SIZE,
        "order_by":      "updated_at",
        "order_type":    "asc",
        "updated_since": f"{window_start.isoformat()}T00:00:00Z",
    }

    tickets = []
    page = 0

    while True:
        page += 1
        if page > MAX_PAGES:
            raise ValueError(
                f"Fönstret {window_start}–{window_end} är för stort (>300 sidor). "
                "Minska fönsterstorlek i month_windows()."
            )

        params = {**base_params, "page": page}
        attempt = 0

        while True:
            attempt += 1
            try:
                r = requests.get(FRESHDESK_API_URL, auth=auth, headers=headers,
                                 params=params, timeout=60)
            except requests.RequestException as ex:
                if attempt >= MAX_RETRIES:
                    raise
                wait = RETRY_BACKOFF ** attempt
                print(f"    Request error, retry {attempt}/{MAX_RETRIES} om {wait}s: {ex}")
                time.sleep(wait)
                continue

            if r.status_code == 401:
                raise RuntimeError("Autentisering misslyckades (401).")
            if r.status_code == 403:
                raise RuntimeError("Åtkomst nekad (403).")
            if r.status_code == 429:
                if attempt >= MAX_RETRIES:
                    r.raise_for_status()
                wait = int(r.headers.get("Retry-After", RETRY_BACKOFF ** attempt))
                print(f"    Rate limited, väntar {wait}s...")
                time.sleep(wait)
                continue
            if r.status_code >= 500:
                if attempt >= MAX_RETRIES:
                    r.raise_for_status()
                time.sleep(RETRY_BACKOFF ** attempt)
                continue
            break

        block = r.json()
        if isinstance(block, dict) and "errors" in block:
            raise RuntimeError(f"Freshdesk API-fel (sida {page}): {block}")
        if not isinstance(block, list):
            raise RuntimeError(f"Oväntat svar (sida {page}): {str(block)[:300]}")

        # Filtrera klientsida: behåll bara tickets med updated_at i fönstret
        window_end_str = window_end.isoformat()
        in_window = [
            extract_fields(t) for t in block
            if (t.get("updated_at") or "")[:10] < window_end_str
        ]
        tickets.extend(in_window)

        # Om alla tickets i sidan är utanför fönstret är vi klara
        if len(block) < PAGE_SIZE or not in_window:
            break

    return tickets

In [4]:
# Cell 4 — Kör backfill månadsvis och spara
api_key = read_api_key()
windows = month_windows(BACKFILL_FROM, BACKFILL_TO)
print(f"{len(windows)} månadsfönster: {windows[0][0]} → {windows[-1][1]}")

all_tickets_by_id: dict[int, dict] = {}

for win_start, win_end in windows:
    print(f"\nFönster {win_start} – {win_end}...")
    chunk = fetch_window(api_key, win_start, win_end)
    print(f"  {len(chunk)} tickets i fönstret")
    for t in chunk:
        tid = t["id"]
        # Behåll den med senast updated_at om ticketen dyker upp i flera fönster
        existing = all_tickets_by_id.get(tid)
        if existing is None or (t["updated_at"] or "") > (existing["updated_at"] or ""):
            all_tickets_by_id[tid] = t

all_tickets = list(all_tickets_by_id.values())
print(f"\nTotalt unika tickets: {len(all_tickets)}")

out_name = f"freshdesk_backfill_{BACKFILL_FROM.strftime('%Y%m%d')}_{BACKFILL_TO.strftime('%Y%m%d')}.json"
out_path = RAW_DIR / out_name
with out_path.open("w", encoding="utf-8") as f:
    json.dump(all_tickets, f, ensure_ascii=False, indent=2)

print(f"Sparade: {out_path}")
print(f"Filstorlek: {out_path.stat().st_size / 1024:.0f} KB")

Läser API-nyckel från: C:\Users\michael.brostrom\Documents\GitHub\statistik-freshdesk-linear\credentials\Freshdesk_API-key.txt
14 månadsfönster: 2025-05-01 → 2026-06-05

Fönster 2025-05-01 – 2025-06-01...
  4917 tickets i fönstret

Fönster 2025-06-01 – 2025-07-01...
  4383 tickets i fönstret

Fönster 2025-07-01 – 2025-08-01...
  4386 tickets i fönstret

Fönster 2025-08-01 – 2025-09-01...
  3213 tickets i fönstret

Fönster 2025-09-01 – 2025-10-01...
  4563 tickets i fönstret

Fönster 2025-10-01 – 2025-11-01...
  4994 tickets i fönstret

Fönster 2025-11-01 – 2025-12-01...
  5593 tickets i fönstret

Fönster 2025-12-01 – 2026-01-01...
  5983 tickets i fönstret

Fönster 2026-01-01 – 2026-02-01...
  4337 tickets i fönstret

Fönster 2026-02-01 – 2026-03-01...
  4823 tickets i fönstret

Fönster 2026-03-01 – 2026-04-01...
  5742 tickets i fönstret

Fönster 2026-04-01 – 2026-05-01...
  6205 tickets i fönstret

Fönster 2026-05-01 – 2026-06-01...
  6359 tickets i fönstret

Fönster 2026-06-01 – 202

In [5]:
# Cell 5 — Verifiering
import pprint

dates = [t["created_at"][:10] for t in all_tickets if t.get("created_at")]
print(f"Antal tickets:      {len(all_tickets)}")
print(f"Äldsta created_at:  {min(dates) if dates else 'N/A'}")
print(f"Senaste created_at: {max(dates) if dates else 'N/A'}")
print(f"Fält: {list(all_tickets[0].keys()) if all_tickets else 'N/A'}")
print("\nFörsta ticketen:")
pprint.pprint(all_tickets[0] if all_tickets else {})

Antal tickets:      69434
Äldsta created_at:  2020-10-14
Senaste created_at: 2026-06-04
Fält: ['id', 'subject', 'status', 'priority', 'created_at', 'updated_at', 'due_by', 'group_id', 'product_id']

Första ticketen:
{'created_at': '2025-04-01T00:30:58Z',
 'due_by': '2026-09-24T15:00:00Z',
 'group_id': 14000111939,
 'id': 251919,
 'priority': 1,
 'product_id': 14000002046,
 'status': 5,
 'subject': 'SDS-Update for Material-No. 110003817 ID 165334',
 'updated_at': '2025-05-01T00:15:14Z'}
